# ADMM-MPC ramp metering

Put this notebook beside `Benchmark_calculation.ipynb` and the two PeMS text files, then restart the kernel and run all cells. Set `PROJECT_DIR` in the first cell if those files are in a different folder. The kernel needs NumPy, pandas, CVXPY, Clarabel, and OSQP.

The notebook runs four closed-loop experiments in sequence. It takes the CTM, PeMS inputs, and evaluation weights directly from the benchmark notebook. Results are held in memory.

In [ ]:
from pathlib import Path

import cvxpy as cp
import numpy as np
import pandas as pd
from IPython.display import display

# Use the folder containing the benchmark notebook and both PeMS text files.
PROJECT_DIR = Path.cwd()
NBK = PROJECT_DIR / "Benchmark_calculation.ipynb"
HORIZONS = (5, 10)
TERMINAL_QUEUE_MINUTES = 30.0
SOLVERS = (cp.CLARABEL, cp.OSQP)

if not NBK.is_file():
    raise FileNotFoundError(f"Set PROJECT_DIR to the folder containing {NBK.name}.")
missing_solvers = set(SOLVERS) - set(cp.installed_solvers())
if missing_solvers:
    raise RuntimeError(f"Install {', '.join(sorted(missing_solvers))} in this notebook's Python environment.")

## Exact CTM and comparison baselines

Apply every meter command through the benchmark CTM. After the two-hour window, reopen all ramps and continue with zero new demand until the remaining inventory clears. Ramp service and ramp backlog are measured at the two-hour mark. The fixed-rate baseline replays the caps recovered from the archived fixed-policy notebook.

In [ ]:
# 2. Exact nonlinear CTM and two-hour benchmark replay.
# NBK is the path to Benchmark_calculation.ipynb from the preceding cell.
from contextlib import redirect_stdout
from io import StringIO
from math import isclose
from pathlib import Path
import json


benchmark_path = Path(NBK).resolve()
notebook = json.loads(benchmark_path.read_text(encoding="utf-8-sig"))
code_sources = [
    (number, "".join(cell["source"]))
    for number, cell in enumerate(notebook["cells"], start=1)
    if cell["cell_type"] == "code"
]
G = {
    "__file__": str(benchmark_path),
    "__name__": "__main__",
    "display": lambda *args, **kwargs: None,
}
unused_plotting_lines = {
    "%matplotlib inline",
    "import matplotlib.pyplot as plt",
    "import seaborn as sns",
}

with redirect_stdout(StringIO()):
    for number, source in code_sources[:6]:
        source = "\n".join(
            line for line in source.splitlines() if line.strip() not in unused_plotting_lines
        )
        # Resolve the raw inputs relative to the benchmark notebook.
        source = source.replace(
            "start = Path.cwd().resolve()",
            "start = Path(__file__).resolve().parent",
        )
        exec(compile(source, f"{benchmark_path} cell {number}", "exec"), G)

    # Initialize the benchmark input mapping and diagnostics in memory.
    number, source = code_sources[6]
    reference_source, marker, _ = source.partition("# 7. Save pickle file")
    if not marker:
        raise ValueError("Could not locate the benchmark export boundary.")
    exec(compile(reference_source, f"{benchmark_path} cell {number}", "exec"), G)


def initial_ctm_state():
    """Return mainline, physical ramp queue, spillback, boundary and entry queues."""
    return (
        G["mainline_initial_state"].copy(),
        G["ramp_queue_0"].copy(),
        G["ramp_overflow_queue_0"].copy(),
        0.0,
        G["external_entry_queue_0"].copy(),
    )


def true_step(state, command, step):
    """Advance the exact CTM. Use step=None for zero-demand clearance."""
    x, ramp_queue, spillback, boundary_queue, entry_queue = state
    dt = G["delta_t"]
    ramp_storage = G["ramp_max_queue_by_u"]
    if step is None:
        arrivals = dict.fromkeys(G["ramp_ids"], 0.0)
        boundary_arrival = 0.0
        external_inflow = dict.fromkeys(G["cell_ids"], 0.0)
        fixed_outflow = external_inflow
    else:
        arrivals = {ramp: G["ramp_arrival_series"][ramp][step] for ramp in G["ramp_ids"]}
        boundary_arrival = G["q_in_boundary_series"][step]
        external_inflow = G["external_inflow_series"][step]
        fixed_outflow = G["fixed_outflow_series"][step]
    available, requested = G["ramp_requested_release_one_step"](
        R_current=ramp_queue,
        B_current=spillback,
        ramp_arrival_step=arrivals,
        commanded_release_step=command,
    )
    (
        next_x, downstream_flow, _, _, _, _, exit_flow, _, _, actual_release,
        _, next_boundary_queue, boundary_tts, _, _, next_entry_queue,
    ) = G["ctm_15sec_step"](
        x_current=x,
        q_in_boundary_step=boundary_arrival,
        external_inflow_step=external_inflow,
        fixed_outflow_step=fixed_outflow,
        requested_release_step=requested,
        inflow_capacity=G["ctm_inflow_capacity_official"],
        outflow_capacity=G["ctm_outflow_capacity_official"],
        physical_capacity=G["physical_capacity"],
        movement_factor_by_cell=G["movement_factor_by_cell_official"],
        wave_speed_ratio_by_cell=G["wave_speed_ratio_by_cell_official"],
        upstream_boundary_queue=boundary_queue,
        exit_split_by_cell=G["exit_split_by_cell"],
        merge_priority=G["MERGE_PRIORITY"],
        external_entry_queue=entry_queue,
    )
    next_ramp_queue, next_spillback, _ = G["ramp_next_queue_after_actual_release"](
        available_demand_step=available,
        actual_release_step=actual_release,
        ramp_max_queue_by_u=ramp_storage,
    )
    _, mainline_tts, _, _ = G["mainline_performance_one_step"](
        x_now=x, x_next=next_x, q_out=downstream_flow, actual_f_out=exit_flow,
        tt_ff_min=G["tt_ff_min"], delta_t=dt,
    )
    _, ramp_tts = G["ramp_delay_with_cap"](
        R_current=ramp_queue, R_next=next_ramp_queue,
        B_current=spillback, B_next=next_spillback,
        ramp_arrival_step=arrivals, commanded_release_step=command,
        requested_release_step=requested, actual_release_step=actual_release, delta_t=dt,
    )
    _, _, _, fairness = G["fairness_penalty_one_step"](
        R_current=ramp_queue, B_current=spillback,
        R_next=next_ramp_queue, B_next=next_spillback,
        ramp_max_queue_by_u=ramp_storage, delta_t=dt, gamma=G["gamma"],
    )
    exposure = G["exposure_penalty_one_step"](
        x_current=x, x_next=next_x, B_current=spillback, B_next=next_spillback,
        safe_threshold_capacity=G["safe_threshold_capacity"], delta_t=dt,
        lambda_safe=G["lambda_safe"], lambda_spillback=G["lambda_spillback"],
    )
    entry_tts = sum(
        0.5 * (entry_queue[cell] + next_entry_queue[cell]) * dt for cell in G["cell_ids"]
    )
    costs = {
        "mainline_tts": mainline_tts,
        "ramp_queue_tts": ramp_tts,
        "upstream_boundary_tts": boundary_tts,
        "external_entry_tts": entry_tts,
        "fairness_penalty": fairness,
        **exposure,
        "raw_objective": (
            G["AW"] * mainline_tts + ramp_tts + boundary_tts + entry_tts + fairness
            + exposure["weighted_safe_penalty"] + exposure["weighted_spillback_penalty"]
        ),
    }
    # Check conservation over the mainline and all external queues.
    before = (
        sum(x.values()) + sum(ramp_queue.values()) + sum(spillback.values())
        + boundary_queue + sum(entry_queue.values())
    )
    after = (
        sum(next_x.values()) + sum(next_ramp_queue.values()) + sum(next_spillback.values())
        + next_boundary_queue + sum(next_entry_queue.values())
    )
    offered = boundary_arrival + sum(external_inflow.values()) + sum(arrivals.values())
    discharged = downstream_flow[G["cell_ids"][-1]] + sum(exit_flow.values())
    if not isclose(before + offered, after + discharged, rel_tol=0.0, abs_tol=1e-7):
        raise AssertionError(f"Vehicle balance failed at step {step}.")
    return (next_x, next_ramp_queue, next_spillback, next_boundary_queue, next_entry_queue), costs


def rollout(command_at_step):
    """Replay the two-hour window. The callback receives (step, state)."""
    state = initial_ctm_state()
    totals = {}
    for step in range(G["num_steps"]):
        state, costs = true_step(state, command_at_step(step, state), step)
        for name, value in costs.items():
            totals[name] = totals.get(name, 0.0) + value
    return state, totals


def fully_open_command(step, state):
    return {ramp: G["commanded_release_series"][ramp][step] for ramp in G["ramp_ids"]}


benchmark_end_state, benchmark_replay_totals = rollout(fully_open_command)
for name, actual in benchmark_replay_totals.items():
    expected = G["official_totals"][name]
    if not isclose(actual, expected, rel_tol=0.0, abs_tol=1e-8):
        raise AssertionError(f"Benchmark replay mismatch for {name}: {actual} vs {expected}")

print(f"Fully open exact CTM replay (480 steps): {benchmark_replay_totals['raw_objective']:.6f}")

In [ ]:
# 3. Replay and score the report's fully open and fixed-rate baselines.
# Run the updated exact CTM cell first.
from math import fsum

import pandas as pd


def system_inventory(state):
    mainline, physical_ramp, spillback, boundary, external = state
    return fsum((
        fsum(mainline.values()), fsum(physical_ramp.values()),
        fsum(spillback.values()), boundary, fsum(external.values()),
    ))


def score_with_clearance(end_state, window_costs, *, tolerance=1e-7, max_steps=480):
    """Apply fully open ramps to the exact CTM until residual traffic clears."""
    _, physical_ramp, spillback, _, _ = end_state
    remaining_on_ramps = fsum(physical_ramp.values()) + fsum(spillback.values())
    _, initial_ramp, initial_spillback, _, _ = initial_ctm_state()
    ramp_demand = fsum((
        fsum(initial_ramp.values()), fsum(initial_spillback.values()),
        fsum(value for series in G["ramp_arrival_series"].values() for value in series),
    ))
    totals = window_costs.copy()
    state = end_state
    steps = 0
    while system_inventory(state) > tolerance:
        if steps >= max_steps:
            raise RuntimeError(f"Traffic did not clear after {max_steps} extra steps.")
        _, physical_ramp, spillback, _, _ = state
        command = {ramp: physical_ramp[ramp] + spillback[ramp] for ramp in G["ramp_ids"]}
        state, costs = true_step(state, command, None)
        for name, cost in costs.items():
            totals[name] += cost
        steps += 1
    return {
        **totals,
        "two_hour_objective": window_costs["raw_objective"],
        "clearance_cost": totals["raw_objective"] - window_costs["raw_objective"],
        "clearance_steps": steps,
        "served_fraction": (ramp_demand - remaining_on_ramps) / ramp_demand if ramp_demand else 1.0,
        "vehicles_left_on_ramps": remaining_on_ramps,
    }


# Caps in vehicles per 15 seconds, recovered from the archived fixed-policy run.
fixed_rates = {"u_4th": 10.975691890625, "u_price": 14.69972097265625,
               "u_mattie": 3.06, "u_avila": 2.219721}
if set(fixed_rates) != set(G["ramp_ids"]):
    raise ValueError("The archived fixed rates do not cover the benchmark's ramp IDs.")

baseline_score = score_with_clearance(benchmark_end_state, benchmark_replay_totals)
fixed_end_state, fixed_window_costs = rollout(lambda step, state: fixed_rates)
fixed_score = score_with_clearance(fixed_end_state, fixed_window_costs)

baselines = pd.DataFrame(
    [baseline_score, fixed_score], index=["Fully open", "Report fixed-rate policy"]
)
baselines["percent_vs_fully_open"] = 100 * (
    baselines["raw_objective"] / baseline_score["raw_objective"] - 1
)
display(baselines[[
    "two_hour_objective", "clearance_cost", "raw_objective", "mainline_tts",
    "ramp_queue_tts", "fairness_penalty", "served_fraction",
    "vehicles_left_on_ramps", "percent_vs_fully_open",
]].round(6))

## Receding-horizon QP

Use 5- and 10-step windows, the benchmark's mainline weight of 1, and a 30-minute terminal queue cost. The forecast uses the PeMS demand for each window, shortened at the end of the two-hour interval. The terminal term covers ramp, upstream boundary, and external-entry queues.

The archived controller implementation was unavailable. This reconstruction relaxes FIFO and merge priorities into shared flow capacities in its planning model. Exact replay uses the benchmark's FIFO, entry, ramp, and merge rules. The report's archived controller scores can differ from this reconstruction. OSQP is the ADMM-based QP solver for the same optimization model.

In [ ]:
S = G["shared_benchmark_inputs"]
cells = tuple(G["cell_ids"])
ramps = tuple(S["ramp_ids"])
steps = S["num_steps"]
dt = S["delta_t"]


def ordered(mapping, keys):
    return np.array([mapping[key] for key in keys], dtype=float)


arrivals = np.array([S["ramp_arrival_series"][ramp] for ramp in ramps])
boundary_demand = np.array(S["q_in_boundary_series"])
entry_demand, fixed_exit_demand = (
    np.array([ordered(row, cells) for row in S[name]]).T
    for name in ("external_inflow_series", "fixed_outflow_series")
)
ramp_to_cell = np.zeros((len(cells), len(ramps)))
for index, ramp in enumerate(ramps):
    ramp_to_cell[cells.index(S["ramp_cell_map"][ramp]), index] = 1.0


def state_vector(state):
    mainline, physical, overflow, boundary, external = state
    return np.r_[
        ordered(mainline, cells), ordered(physical, ramps) + ordered(overflow, ramps),
        boundary, ordered(external, cells),
    ]


class WindowQP:
    """Capacity-constrained planning model. The benchmark CTM applies the commands."""

    def __init__(self, horizon):
        n, r, h = len(cells), len(ramps), horizon
        self.horizon = h
        self.initial = cp.Parameter(2 * n + r + 1, nonneg=True)
        self.ramp_arrivals = cp.Parameter((r, h), nonneg=True)
        self.boundary_arrivals = cp.Parameter(h, nonneg=True)
        self.entry_arrivals = cp.Parameter((n, h), nonneg=True)
        self.fixed_exits = cp.Parameter((n, h), nonneg=True)

        x = cp.Variable((n, h + 1), nonneg=True)
        w = cp.Variable((r, h + 1), nonneg=True)
        upstream = cp.Variable(h + 1, nonneg=True)
        entry = cp.Variable((n, h + 1), nonneg=True)
        total_send = cp.Variable((n, h), nonneg=True)
        fixed_exit = cp.Variable((n, h), nonneg=True)
        from_boundary = cp.Variable(h, nonneg=True)
        from_entry = cp.Variable((n, h), nonneg=True)
        release = cp.Variable((r, h), nonneg=True)
        self.release = release

        storage = ordered(S["physical_capacity"], cells)[:, None]
        safe_storage = ordered(S["safe_threshold_capacity"], cells)[:, None]
        ramp_storage = ordered(S["ramp_max_queue_by_u"], ramps)[:, None]
        sending_limit = ordered(S["outflow_capacity"], cells)[:, None]
        receiving_limit = ordered(S["inflow_capacity"], cells)[:, None]
        free_flow = ordered(S["movement_factor_by_cell"], cells)[:, None]
        wave = ordered(S["wave_speed_ratio_by_cell"], cells)[:, None]
        continuing_fraction = 1.0 - ordered(S["exit_split_by_cell"], cells)[:, None]
        continuing = cp.multiply(continuing_fraction, total_send - fixed_exit)
        inflow = (
            cp.vstack([cp.reshape(from_boundary, (1, h), order="C"), continuing[:-1, :]])
            + from_entry + ramp_to_cell @ release
        )

        constraints = [
            x[:, 0] == self.initial[:n],
            w[:, 0] == self.initial[n:n + r],
            upstream[0] == self.initial[n + r],
            entry[:, 0] == self.initial[n + r + 1:],
            x <= storage,
            total_send <= sending_limit,
            total_send <= cp.multiply(free_flow, x[:, :-1]),
            fixed_exit <= total_send,
            fixed_exit <= self.fixed_exits,
            from_boundary <= upstream[:-1] + self.boundary_arrivals,
            upstream[1:] == upstream[:-1] + self.boundary_arrivals - from_boundary,
            from_entry <= entry[:, :-1] + self.entry_arrivals,
            entry[:, 1:] == entry[:, :-1] + self.entry_arrivals - from_entry,
            release <= w[:, :-1] + self.ramp_arrivals,
            w[:, 1:] == w[:, :-1] + self.ramp_arrivals - release,
            inflow <= receiving_limit,
            inflow <= cp.multiply(wave, storage - x[:, :-1]),
            x[:, 1:] == x[:, :-1] + inflow - total_send,
        ]

        def midpoint(history):
            return 0.5 * (history[..., :-1] + history[..., 1:])

        mean_x = midpoint(x)
        mean_w = midpoint(w)
        stress = cp.multiply(1.0 / ramp_storage, mean_w)
        center = np.eye(r) - np.ones((r, r)) / r
        fairness = S["gamma"] * float(np.mean(ramp_storage)) * cp.sum_squares(center @ stress)
        safety = S["lambda_safe"] * cp.sum(cp.pos(mean_x - safe_storage))
        overflow = S["lambda_spillback"] * cp.sum(midpoint(cp.pos(w - ramp_storage)))
        stage_cost = dt * (
            S["AW"] * cp.sum(mean_x) + cp.sum(mean_w) + cp.sum(midpoint(upstream))
            + cp.sum(midpoint(entry)) + fairness + safety + overflow
        )
        terminal_cost = TERMINAL_QUEUE_MINUTES * (
            cp.sum(w[:, -1]) + upstream[-1] + cp.sum(entry[:, -1])
        )
        self.problem = cp.Problem(cp.Minimize(stage_cost + terminal_cost), constraints)
        if not self.problem.is_qp() or not self.problem.is_dpp():
            raise ValueError("The configured window must be a reusable convex QP.")

    def solve(self, state, step, solver):
        upcoming = slice(step, step + self.horizon)
        self.initial.value = state_vector(state)
        self.ramp_arrivals.value = arrivals[:, upcoming]
        self.boundary_arrivals.value = boundary_demand[upcoming]
        self.entry_arrivals.value = entry_demand[:, upcoming]
        self.fixed_exits.value = fixed_exit_demand[:, upcoming]
        clarabel = dict(tol_gap_abs=5e-6, tol_gap_rel=5e-6, tol_feas=5e-6)
        if solver == cp.CLARABEL:
            attempts = [(cp.CLARABEL, clarabel, True), (cp.CLARABEL, clarabel, False)]
        else:
            osqp = dict(eps_abs=5e-6, eps_rel=5e-6, max_iter=10_000, polishing=True)
            attempts = [
                (cp.OSQP, osqp, True),
                (cp.OSQP, {**osqp, "max_iter": 20_000, "rho": 0.2,
                           "adaptive_rho_interval": 25, "scaled_termination": True}, False),
                (cp.CLARABEL, clarabel, False),
            ]
        failures = []
        for attempt, (backend, options, warm_start) in enumerate(attempts):
            try:
                self.problem.solve(solver=backend, warm_start=warm_start, **options)
            except cp.error.SolverError as error:
                failures.append(str(error))
                continue
            status = self.problem.status
            if status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
                failures.append(status)
                continue
            values = self.release.value
            if values is None or not np.all(np.isfinite(values)) or not np.isfinite(self.problem.value):
                failures.append("nonfinite solution")
                continue
            max_violation = max(
                float(np.max(np.abs(constraint.violation())))
                for constraint in self.problem.constraints
            )
            # This threshold applies to the forecast QP. The applied command is
            # independently checked against the exact plant at much tighter tolerance.
            acceptance_tolerance = 0.005 if backend == cp.OSQP else 2e-5
            if max_violation <= acceptance_tolerance:
                command = np.maximum(values[:, 0], 0.0)
                return dict(zip(ramps, command)), {
                    "status": status, "iterations": self.problem.solver_stats.num_iters,
                    "max_qp_constraint_violation": max_violation,
                    "retried": attempt > 0,
                    "solver_used": backend,
                    "fallback": backend != solver,
                }
            failures.append(f"maximum QP constraint violation {max_violation:.3g}")
        raise RuntimeError(f"{solver} failed at step {step}, window {self.horizon}: {'; '.join(failures)}")

## Closed-loop runs

At each step, solve the window, apply its first meter command, and use the resulting exact state to initialize the next window. Recheck all commands with the benchmark simulator before adding clearance. If OSQP cannot return an acceptable solution after a retry, solve that window with Clarabel and count the fallback in the diagnostics.

In [ ]:
from functools import lru_cache


def replay_commands(commands):
    """Run the benchmark's native simulator with the online command schedule."""
    fields = (
        "num_steps", "mainline_initial_state", "ramp_queue_0", "q_in_boundary_series",
        "ramp_arrival_series", "external_inflow_series", "fixed_outflow_series",
        "inflow_capacity", "outflow_capacity", "physical_capacity", "safe_threshold_capacity",
        "ramp_name_map", "ramp_max_queue_named", "ramp_max_queue_by_u", "tt_ff_min",
        "delta_t", "gamma", "lambda_safe", "lambda_spillback", "AW", "movement_factor_by_cell",
        "wave_speed_ratio_by_cell", "exit_split_by_cell", "ramp_overflow_queue_0",
        "external_entry_queue_0",
    )
    return G["simulate_state_based_benchmark_480_steps"](
        **{name: S[name] for name in fields}, commanded_release_series=commands,
    )


def exact_checks(history, online_states):
    native_states = np.array([state_vector(state) for state in zip(
        history["x"], history["R"], history["B"],
        history["upstream_boundary_queue"], history["external_entry_queue"],
    )])
    checks = {"online_replay_state_error": float(np.max(np.abs(online_states - native_states)))}
    for name in (
        "mainline_mass_residual", "upstream_boundary_mass_residual", "external_entry_mass_residual",
        "max_receiving_violation", "max_sending_violation", "max_merge_violation",
    ):
        checks[name] = abs(G[f"compute_{name}"](history))
    conservation = [
        S["ramp_queue_0"][ramp] + S["ramp_overflow_queue_0"][ramp] + fsum(S["ramp_arrival_series"][ramp])
        - fsum(step[ramp] for step in history["actual_release"])
        - history["R_final"][ramp] - history["B_final"][ramp]
        for ramp in ramps
    ]
    checks["max_ramp_mass_residual"] = max(map(abs, conservation))
    if any(not np.isfinite(value) or value > 2e-7 for value in checks.values()):
        raise AssertionError(f"Exact replay checks failed: {checks}")
    return checks


def run_mpc(horizon, solver):
    # Each run caches only the few window sizes it uses. Runs execute serially.
    window = lru_cache(maxsize=horizon)(WindowQP)
    state = initial_ctm_state()
    states = []
    commands = {ramp: [] for ramp in ramps}
    diagnostics = []
    for step in range(steps):
        problem = window(min(horizon, steps - step))
        command, info = problem.solve(state, step, solver)
        state, _ = true_step(state, command, step)
        states.append(state_vector(state))
        diagnostics.append(info)
        for ramp, value in command.items():
            commands[ramp].append(float(value))

    history = replay_commands(commands)
    checks = exact_checks(history, np.array(states))
    window_costs = {
        name: fsum(history["total_objective" if name == "raw_objective" else name])
        for name in benchmark_replay_totals
    }
    score = score_with_clearance(state, window_costs)
    return score, pd.DataFrame(diagnostics), checks, history


results = {}
for horizon in HORIZONS:
    for solver in SOLVERS:
        solver_label = "Clarabel" if solver == cp.CLARABEL else "OSQP-first (ADMM)"
        label = f"K={horizon} / {solver_label}"
        print(f"Running {label} ({steps} short solves)", flush=True)
        results[label] = run_mpc(horizon, solver)


comparison = pd.DataFrame.from_dict({
    "Fully open": baseline_score,
    "Report fixed-rate policy": fixed_score,
    **{name: result[0] for name, result in results.items()},
}, orient="index")
comparison["percent_vs_fully_open"] = 100.0 * (
    comparison["raw_objective"] / baseline_score["raw_objective"] - 1.0
)
display(comparison[[
    "raw_objective", "percent_vs_fully_open", "mainline_tts", "ramp_queue_tts",
    "fairness_penalty", "served_fraction", "vehicles_left_on_ramps",
    "two_hour_objective", "clearance_cost",
]].round(6))

## Solver and release checks

Solver diagnostics cover every online solve. The release audit counts commands, available requests, actual accepted releases, and the ramp backlog at the end of the two-hour window.

In [ ]:
solver_rows = []
release_rows = []
for policy, (_, diagnostics, checks, history) in results.items():
    solver_rows.append({
        "policy": policy,
        "online_solves": len(diagnostics),
        "optimal_inaccurate": int((diagnostics["status"] == cp.OPTIMAL_INACCURATE).sum()),
        "retried": int(diagnostics["retried"].sum()),
        "clarabel_fallbacks": int(diagnostics["fallback"].sum()),
        "max_solver_iterations": diagnostics["iterations"].max(),
        "max_qp_constraint_violation": diagnostics["max_qp_constraint_violation"].max(),
        **checks,
    })
    for ramp in ramps:
        release_rows.append({
            "policy": policy,
            "ramp": ramp,
            "commanded": fsum(row[ramp] for row in history["commanded_release"]),
            "requested": fsum(row[ramp] for row in history["requested_release"]),
            "actual_release": fsum(row[ramp] for row in history["actual_release"]),
            "left_at_two_hours": history["R_final"][ramp] + history["B_final"][ramp],
        })

display(pd.DataFrame(solver_rows).set_index("policy"))
display(pd.DataFrame(release_rows).set_index(["policy", "ramp"]).round(6))